In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Cloud Storage
from google.cloud import storage
storage_client = storage.Client(project='LLAMA')


In [3]:
!git clone --branch feature/turboquant-kv-cache https://github.com/TheTom/llama-cpp-turboquant.git
%cd llama-cpp-turboquant

!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j$(nproc)

Cloning into 'llama-cpp-turboquant'...
remote: Enumerating objects: 93425, done.
remote: Total 93425 (delta 0), reused 0 (delta 0), pack-reused 93425 (from 1)
Receiving objects: 100% (93425/93425), 376.14 MiB | 27.00 MiB/s, done.
Resolving deltas: 100% (64433/64433), done.
/kaggle/working/llama-cpp-turboquant
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIB

In [4]:
!nvidia-smi
!which nvcc || true
!find /usr/local /usr/lib /lib -name 'libcuda.so*' 2>/dev/null | head -30

Sun Sep 13 22:41:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!cmake -B build \
    -DGGML_CUDA=ON \
    -DCMAKE_BUILD_TYPE=Release \
    -DCUDAToolkit_ROOT=/usr/local/cuda

CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75-real CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.18.1
-- ggml commit:  407f3237b
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.6s)
CMake Error at ggml/src/ggml-cuda/CMakeLists.txt:207 (target_link_libraries):
  Target "ggml-cuda" links to:

    CUDA::cuda_driver

  but the target was not found.  Possible reasons include:

    * There is a typo in the target name.
    * A find_package call is missing for an IMPORTED target.
    * An ALIAS target is missing.



-- Generating done (0.4s)
CMake Generat

In [6]:
!cmake --build build -j$(nproc)

[  0%] Built target sha256
[  0%] Built target llama-common-base
[  0%] Built target cpp-httplib
[  2%] Built target ggml-base
[  3%] Built target xxhash
[  3%] Built target sha1
[  3%] Built target llama-llava-cli
[  3%] Built target llama-gemma3-cli
[  3%] Built target llama-ui-embed
[  3%] Built target llama-minicpmv-cli
[  4%] Built target llama-qwen2vl-cli
[  5%] Provisioning UI assets
[  7%] Built target ggml-cpu
-- UI: npm output up-to-date, skipping build
[  7%] Linking CUDA shared library ../../../bin/libggml-cuda.so
/usr/bin/ld: cannot find -lCUDA::cuda_driver: No such file or directory
collect2: error: ld returned 1 exit status
gmake[2]: *** [ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/build.make:3323: bin/libggml-cuda.so.0.18.1] Error 1
gmake[1]: *** [CMakeFiles/Makefile2:2515: ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/all] Error 2
gmake[1]: *** Waiting for unfinished jobs....
-- UI: gzip compression applied (/kaggle/working/llama-cpp-turboquant/build/tools/ui/dist/_gzip)